In [1]:
import sys
import os
import mysql.connector
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Ambil konfigurasi database dari folder utama project kalian
sys.path.append(os.path.abspath('..'))
from config import get_db_config

config = get_db_config()
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)

db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)

print(f"✅ Terhubung ke DB_NEW hasil migrasi : {config['db_new']['database']}")
print(f"🔮 Terhubung ke DB_FUTURE blueprint  : {config['db_future']['database']}")

✅ Terhubung ke DB_NEW hasil migrasi : dataleap_v5_migration
🔮 Terhubung ke DB_FUTURE blueprint  : 2


In [3]:
print("================================================================================")
print(" 🕵️‍♂️ RADAR GLOBAL: DETEKTOR PERBEDAAN DAFTAR TABEL [ DB_NEW VS DB_FUTURE ] 🕵️‍♂️ ")
print("================================================================================")

try:
    # 1. Ambil daftar seluruh tabel asli yang ada di DB_NEW saat ini
    cursor_new.execute("SHOW TABLES")
    tables_in_new = set([list(row.values())[0] for row in cursor_new.fetchall()])
    
    # 2. Ambil daftar seluruh tabel asli yang ada di DB_FUTURE saat ini
    cursor_future.execute("SHOW TABLES")
    tables_in_future = set([list(row.values())[0] for row in cursor_future.fetchall()])
    
    # 3. Hitung Operasi Himpunan Matematika (Set Difference) untuk mencari selisih
    tabel_baru_di_future = sorted(list(tables_in_future - tables_in_new))
    tabel_ekstra_di_new  = sorted(list(tables_in_new - tables_in_future))
    tabel_yang_klop      = sorted(list(tables_in_new & tables_in_future))
    
    # ----------------------------------------------------------------------------
    # 📊 CETAK PAPAN RINGKASAN STRATEGIS DI PALING ATAS (ANTI-SCROLL SUMMARY BOARD)
    # ----------------------------------------------------------------------------
    print("\n================================================================================")
    print(" 📊 PAPAN AUDIT KESELARASAN JUMLAH TABEL GLOBAL (GLOBAL SUMMARY BOARD) 📊")
    print("================================================================================")
    print(f"📦 Total Tabel Terdeteksi di DB_NEW    : {len(tables_in_new)} Tabel")
    print(f"🔮 Total Tabel Terdeteksi di DB_FUTURE : {len(tables_in_future)} Tabel")
    print("-" * 80)
    
    if len(tabel_baru_di_future) == 0 and len(tabel_ekstra_di_new) == 0:
        print("✨ STATUS DATABASE: PERFECT MATCH! ✨")
        print("🟢 Seluruh nama tabel di DB_NEW sudah sama persis 100% dengan skema target DB_FUTURE!")
    else:
        print("⚠️  STATUS DATABASE: TERDETEKSI PERBEDAAN DAFTAR NAMA TABEL! ⚠️")
        if tabel_baru_di_future:
            print(f"🚨 Ada {len(tabel_baru_di_future)} tabel BARU di DB_FUTURE yang BELUM kalian buat di DB_NEW.")
        if tabel_ekstra_di_new:
            print(f"🗑️  Ada {len(tabel_ekstra_di_new)} tabel TERTINGGAL di DB_NEW yang SUDAH DIHAPUS di DB_FUTURE.")
    print("================================================================================\n")

    # ----------------------------------------------------------------------------
    # DETEKSI VISUALISASI DETIL (DETAIL TABLE LISTS DISPLAY)
    # ----------------------------------------------------------------------------
    print("="*80)
    print("🔎 LAPORAN RINCIAN PERBEDAAN STRUKTUR TABEL")
    print("="*80)
    
    # A. Cetak Tabel Baru yang Belum Dimigrasi
    print(f"\n🚨 A. TABEL BARU DI DB_FUTURE (Belum ada di DB_NEW / Harus Segera Dibuat):")
    print("-" * 80)
    if tabel_baru_di_future:
        df_missing = pd.DataFrame(tabel_baru_di_future, columns=['Nama Tabel Baru (Target Blueprint)'])
        display(df_missing)
    else:
        print("✅ Bersih! Tidak ada tabel baru yang tertinggal.")
        
    # B. Cetak Tabel Siluman yang Lupa Dihapus
    print(f"\n🗑️  B. TABEL EXTRA DI DB_NEW (Sudah Dihapus di DB_FUTURE / Layak Di-drop):")
    print("-" * 80)
    if tabel_ekstra_di_new:
        df_extra = pd.DataFrame(tabel_ekstra_di_new, columns=['Nama Tabel Sampah/Lama (Sisa Migrasi)'])
        display(df_extra)
    else:
        print("✅ Bersih! Tidak ada tabel sampah/lama yang tertinggal di database baru.")

    # C. Cetak Tabel yang Sudah Sukses Klop Nama
    print(f"\n🟢 C. DAFTAR TABEL YANG SUDAH MATCH NAMA (Aman Terdaftar di Kedua DB):")
    print("-" * 80)
    df_match = pd.DataFrame(tabel_yang_klop, columns=['Nama Tabel (Sudah Sinkron)'])
    # Menggunakan head(10) agar daftar tabel yang sudah aman tidak terlalu memenuhi layar notebook
    print(f"ℹ️ Menampilkan 10 sampel dari total {len(df_match)} tabel yang sukses terpetakan:")
    display(df_match)
    
    print("\n" + "="*80)
    print("🏁 PROSES AUDIT DAN MAPPING PERBEDAAN TABEL GLOBAL SELESAI VIA RADAR 🏁")
    print("="*80)

except Exception as e:
    print(f"❌ Gagal menjalankan radar detektor global. Alasan: {e}")

 🕵️‍♂️ RADAR GLOBAL: DETEKTOR PERBEDAAN DAFTAR TABEL [ DB_NEW VS DB_FUTURE ] 🕵️‍♂️ 

 📊 PAPAN AUDIT KESELARASAN JUMLAH TABEL GLOBAL (GLOBAL SUMMARY BOARD) 📊
📦 Total Tabel Terdeteksi di DB_NEW    : 104 Tabel
🔮 Total Tabel Terdeteksi di DB_FUTURE : 113 Tabel
--------------------------------------------------------------------------------
⚠️  STATUS DATABASE: TERDETEKSI PERBEDAAN DAFTAR NAMA TABEL! ⚠️
🚨 Ada 9 tabel BARU di DB_FUTURE yang BELUM kalian buat di DB_NEW.

🔎 LAPORAN RINCIAN PERBEDAAN STRUKTUR TABEL

🚨 A. TABEL BARU DI DB_FUTURE (Belum ada di DB_NEW / Harus Segera Dibuat):
--------------------------------------------------------------------------------


,Nama Tabel Baru (Target Blueprint)
0,calon_siswa_fo_detail
1,calon_siswa_form_program_requirements
2,calon_siswa_form_programs
3,calon_siswa_proses_logs
4,catatan_remidi_siswa
5,penilaian_kinerja
6,rapor_setting_kursus
7,siswa_bulk_edit_logs
8,siswa_keluar_feedbacks



🗑️  B. TABEL EXTRA DI DB_NEW (Sudah Dihapus di DB_FUTURE / Layak Di-drop):
--------------------------------------------------------------------------------
✅ Bersih! Tidak ada tabel sampah/lama yang tertinggal di database baru.

🟢 C. DAFTAR TABEL YANG SUDAH MATCH NAMA (Aman Terdaftar di Kedua DB):
--------------------------------------------------------------------------------
ℹ️ Menampilkan 10 sampel dari total 104 tabel yang sukses terpetakan:


,Nama Tabel (Sudah Sinkron)
0,absensi
1,activity_log
2,admin_sarpras
3,bidang_kategori
4,bidang_link
...,...
99,verifikasi_absensi
100,verifikasi_izin
101,verifikasi_surat_keluar
102,web_berita



🏁 PROSES AUDIT DAN MAPPING PERBEDAAN TABEL GLOBAL SELESAI VIA RADAR 🏁
